In [1]:
import json
import pandas as pd

## 1. Lecture et normalisation JSON

Chargement de `restaurants.json` et conversion en DataFrame avec `pd.json_normalize()`, en conservant les métadonnées `ville` et `pays`.

In [2]:
with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
df

,nom,adresse,type,notes,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie


## Challenge 1 — Normalisation d'un JSON imbriqué

Extraction et analyse des restaurants de plusieurs villes italiennes à partir du fichier `villes.json`.

### Objectifs
- Aplatir un JSON imbriqué avec `pd.json_normalize()`.
- Conserver les informations des villes avec `meta`.
- Calculer la colonne `note_moyenne` à partir de `notes`.
- Identifier la ville ayant la meilleure moyenne avec `villes.nom`.
- Compter les restaurants par catégorie avec `type`.

Étape 1 — explorer sans pandas

In [3]:
with open("../data/villes.json", encoding="utf-8") as f:
    data = json.load(f)

print("Nombre de villes :", len(data["villes"]))

print("La deuxième ville est :", data["villes"][1]["nom"])

nombre_restaurant_Syracuse = next(
    (len(x["restaurants"]) for x in data["villes"] if x["nom"] == "Syracuse"),
    None
)
print("Nombre de restaurant à Syracuse :", nombre_restaurant_Syracuse)

for villes in data["villes"]:
    if villes["nom"] == "Palerme":
        if villes["restaurants"]:
            print("Premier restaurant à Palerme :",
                villes["restaurants"][0]["nom"]
            )
        else:
            print("Aucun Restaurant à Palerme")
        break

Nombre de villes : 3
La deuxième ville est : Palerme
Nombre de restaurant à Syracuse : 2
Premier restaurant à Palerme : Antica Focacceria


Étape 2 — compter à la main

In [4]:
for villes in data["villes"]:
    print(villes["nom"], ":", len(villes.get("restaurants", [])))

Catane : 3
Palerme : 2
Syracuse : 2


Étape 3 — tout mettre à plat

In [ ]:
df = pd.json_normalize(
    data,
    record_path=["villes", "restaurants"],
    meta=[
        ["villes", "nom"],
        ["villes", "region"],
        ["villes", "habitants"]
    ]
)

Étape 3 — tout mettre à plat

In [ ]:
df["note_moyenne"] = df["notes"].apply(lambda x: (sum(x) / len(x)))

print("Ville ayant la meilleure moyenne des restaurants")
moyennes_par_ville = df.groupby("villes.nom")["note_moyenne"].mean().round(2)
meilleure_ville = moyennes_par_ville.nlargest(1)
nom_ville = meilleure_ville.index.item()
note = meilleure_ville.iloc[0]
print(nom_ville, ":", note)

print()
print("Ville ayant la meilleure moyenne des évaluations")
df_notes = df.explode("notes")
df_notes["notes"] = df_notes["notes"].astype("float")

moyennes_evaluations_par_ville = df_notes.groupby("villes.nom")["notes"].mean().round(2)
meilleure_ville = moyennes_evaluations_par_ville.nlargest(1)
nom_ville = meilleure_ville.index.item()
note = meilleure_ville.iloc[0]
print(nom_ville, ":", note)

print()
print("Types de restaurant")
print(df.groupby("type").size())


Ville ayant la meilleure moyenne des restaurants
Syracuse : 4.47

Ville ayant la meilleure moyenne des évaluations
Syracuse : 4.64

Types de restaurant
type
glacier      2
poisson      2
trattoria    3
dtype: int64
